# Cross-dataset shift — calibrate PanNuke, test NuInsSeg (CPU)

Total-count conformal. **Calibrate on PanNuke (source) → test on NuInsSeg (target).**
Real distribution shift (different dataset, 31 organs, model trained only on PanNuke).

**Attach datasets:**
- `hipinhththu/sam3-q1-multiseed-ckpts` — has `phase_C_preds_seed42.pkl` (PanNuke)
- the dataset / notebook-output holding `phase_E_nuinsseg_preds.pkl` (NuInsSeg, from Phase E)

## 00 — Setup

In [ ]:
import os, glob, json, pickle
import numpy as np
print("numpy:", np.__version__)

## 01 — conformal.py (baked)

In [ ]:
%%writefile conformal.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import numpy as np

def empirical_quantile(scores: np.ndarray, alpha: float) -> float:
    n = len(scores)
    if n == 0:
        return float("inf")
    level = np.ceil((n + 1) * (1 - alpha)) / n
    level = min(level, 1.0)
    return float(np.quantile(scores, level, method="higher"))

def pb_count(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    return (scores[:, None] * probs).sum(axis=0)

def pb_variance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    w = scores[:, None] * probs
    return (w * (1.0 - w)).sum(axis=0)

def pb_covariance(scores: np.ndarray, probs: np.ndarray) -> np.ndarray:
    K = probs.shape[1]
    cov = np.zeros((K, K))
    for j in range(K):
        for k in range(K):
            delta = 1.0 if j == k else 0.0
            cov[j, k] = (scores * probs[:, j] * (delta - probs[:, k])).sum()
    return cov

class MarginalSplitConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "MarginalSplitConformal":
        K = cal_gt[0].shape[0]
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                
                for k in range(K):
                    scores_per_class[k].append(float(gt[k]))
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                err = abs(gt[k] - n_pred[k]) / sigma[k]
                scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]), self.alpha)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

class AdaptiveConformalInference:
    def __init__(self, alpha_target: float = 0.1, gamma: float = 0.05):
        self.alpha_target = alpha_target
        self.gamma = gamma
        self.alpha_t = alpha_target
        self.history_q: List[float] = []
        self.history_scores: List[float] = []  

    def reset(self):
        self.alpha_t = self.alpha_target
        self.history_scores = []

    def update(self, score_t: float, covered_t: bool):
        self.history_scores.append(score_t)
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + self.gamma * (self.alpha_target - err_t)
        
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

    def get_quantile(self) -> float:
        if not self.history_scores:
            return 1.0
        return empirical_quantile(np.array(self.history_scores), self.alpha_t)

class ShiftAwareACI(AdaptiveConformalInference):
    def __init__(self, alpha_target: float = 0.1, gamma_0: float = 0.05,
                 lambda_: float = 3.0, gamma_max: float = 0.15):
        super().__init__(alpha_target, gamma_0)
        self.gamma_0 = gamma_0
        self.lambda_ = lambda_
        self.gamma_max = gamma_max
        self.gamma_t_last = gamma_0

    def update(self, score_t: float, covered_t: bool, delta_t: float = 0.0):
        self.history_scores.append(score_t)
        gamma_t = self.gamma_0 * (1.0 + self.lambda_ * max(0.0, delta_t))
        gamma_t = min(gamma_t, self.gamma_max)
        self.gamma_t_last = gamma_t
        err_t = 0.0 if covered_t else 1.0
        self.alpha_t = self.alpha_t + gamma_t * (self.alpha_target - err_t)
        self.alpha_t = max(1e-3, min(0.5, self.alpha_t))

class RollingShiftDetector:
    def __init__(self, window: int = 100, cap: float = 1.0):
        self.window = window
        self.cap = cap
        self.baseline: Optional[float] = None
        self.recent: List[float] = []

    def fit_baseline(self, cal_scores) -> "RollingShiftDetector":
        self.baseline = float(np.median(np.asarray(cal_scores))) + 1e-6
        return self

    def step(self, score_t: float) -> float:
        self.recent.append(float(score_t))
        if len(self.recent) > self.window:
            self.recent.pop(0)
        cur = float(np.median(self.recent))
        delta = (cur - self.baseline) / self.baseline
        return float(np.clip(delta, 0.0, self.cap))

class PBAwareJointConformalOnline:
    def __init__(self, alpha: float = 0.1, window: int = 300):
        self.alpha = alpha
        self.window = window
        self.scores: List[float] = []

    def warmstart(self, cal_scores) -> "PBAwareJointConformalOnline":
        self.scores = list(np.asarray(cal_scores)[-self.window:])
        return self

    def get_quantile(self) -> float:
        if not self.scores:
            return float("inf")
        return empirical_quantile(np.asarray(self.scores[-self.window:]), self.alpha)

    def update(self, score_t: float):
        self.scores.append(float(score_t))
        if len(self.scores) > self.window:
            self.scores = self.scores[-self.window:]

def local_coverage_stats(covered_list, window: int = 100) -> Dict[str, float]:
    c = np.asarray(covered_list, dtype=float)
    n = len(c)
    if n == 0:
        return {"min_local_cov": 0.0, "max_miss_run": 0}
    if n >= window:
        roll = np.convolve(c, np.ones(window) / window, mode="valid")
        min_local = float(roll.min())
    else:
        min_local = float(c.mean())
    run = mx = 0
    for v in covered_list:
        run = 0 if v else run + 1
        mx = max(mx, run)
    return {"min_local_cov": min_local, "max_miss_run": int(mx)}

class PBAwareJointConformal:
    def __init__(self, alpha: float = 0.1):
        self.alpha = alpha
        self.q: float = 0.0

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "PBAwareJointConformal":
        scores = []
        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            K = len(gt)
            if len(s) == 0:
                
                sigma_eps = 1.0
                S_t = max(abs(gt[k]) / sigma_eps for k in range(K))
            else:
                n_pred = pb_count(s, p)
                sigma = np.sqrt(pb_variance(s, p) + 1e-6)
                S_t = max(abs(gt[k] - n_pred[k]) / sigma[k] for k in range(K))
            scores.append(S_t)
        self.q = empirical_quantile(np.array(scores), self.alpha)
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        s = pred["scores"]
        p = pred["probs"]
        K = pred.get("K", 5)
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q * sigma)
        upper = n_pred + self.q * sigma
        return lower, upper

class ClassStratifiedConformal:
    def __init__(self, alpha: float = 0.1, bonferroni: bool = True):
        self.alpha = alpha
        self.bonferroni = bonferroni
        self.q_per_class: Optional[np.ndarray] = None

    def fit(self, cal_preds: List[Dict], cal_gt: List[np.ndarray]) -> "ClassStratifiedConformal":
        K = cal_gt[0].shape[0]
        alpha_eff = self.alpha / K if self.bonferroni else self.alpha
        scores_per_class = [[] for _ in range(K)]

        for pred, gt in zip(cal_preds, cal_gt):
            s = pred["scores"]
            p = pred["probs"]
            if len(s) == 0:
                continue
            n_pred = pb_count(s, p)
            sigma = np.sqrt(pb_variance(s, p) + 1e-6)
            for k in range(K):
                if gt[k] > 0:  
                    err = abs(gt[k] - n_pred[k]) / sigma[k]
                    scores_per_class[k].append(err)

        self.q_per_class = np.array([
            empirical_quantile(np.array(scores_per_class[k]) if scores_per_class[k]
                              else np.array([1.0]), alpha_eff)
            for k in range(K)
        ])
        return self

    def predict_interval(self, pred: Dict) -> Tuple[np.ndarray, np.ndarray]:
        K = len(self.q_per_class)
        s = pred["scores"]
        p = pred["probs"]
        if len(s) == 0:
            return np.zeros(K), np.zeros(K)
        n_pred = pb_count(s, p)
        sigma = np.sqrt(pb_variance(s, p) + 1e-6)
        lower = np.maximum(0, n_pred - self.q_per_class * sigma)
        upper = n_pred + self.q_per_class * sigma
        return lower, upper

def coverage_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                       gt_counts: np.ndarray) -> np.ndarray:
    covered = (gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)
    return covered.mean(axis=0)

def joint_coverage(intervals_lo: np.ndarray, intervals_hi: np.ndarray,
                   gt_counts: np.ndarray) -> float:
    covered_all = ((gt_counts >= intervals_lo) & (gt_counts <= intervals_hi)).all(axis=1)
    return float(covered_all.mean())

def avg_width_per_class(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> np.ndarray:
    return (intervals_hi - intervals_lo).mean(axis=0)

def macro_width(intervals_lo: np.ndarray, intervals_hi: np.ndarray) -> float:
    return float(avg_width_per_class(intervals_lo, intervals_hi).mean())

def split_calibration_test(preds: List[Dict], gts: List[np.ndarray],
                           cal_ratio: float = 0.5,
                           seed: int = 42) -> Tuple[List, List, List, List]:
    n = len(preds)
    rng = np.random.RandomState(seed)
    indices = rng.permutation(n)
    n_cal = int(n * cal_ratio)
    cal_idx = indices[:n_cal]
    test_idx = indices[n_cal:]
    cal_preds = [preds[i] for i in cal_idx]
    cal_gt = [gts[i] for i in cal_idx]
    test_preds = [preds[i] for i in test_idx]
    test_gt = [gts[i] for i in test_idx]
    return cal_preds, cal_gt, test_preds, test_gt

In [ ]:
import sys
if "." not in sys.path: sys.path.insert(0, ".")
from conformal import (AdaptiveConformalInference, PBAwareJointConformalOnline,
                       empirical_quantile, pb_count, pb_variance)
print("conformal loaded.")

## 02 — Load both pkls; PanNuke -> total count

In [ ]:
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None

pan_path = find("phase_C_preds_seed42.pkl")
nu_path  = find("phase_E_nuinsseg_preds.pkl")
assert pan_path, "phase_C_preds_seed42.pkl not found - attach sam3-q1-multiseed-ckpts"
assert nu_path,  "phase_E_nuinsseg_preds.pkl not found - attach the Phase E output dataset"
print("PanNuke:", pan_path)
print("NuInsSeg:", nu_path)

with open(pan_path, "rb") as f: dpan = pickle.load(f)
with open(nu_path, "rb") as f:  dnu  = pickle.load(f)

# PanNuke per-class -> TOTAL count (K=1): total pred = sum_i s_i ; total GT = sum over 5 classes
pan_src = dpan["predictions_by_setting"]["in_dist"]
pan_gtc = np.asarray(dpan["gt_counts"])
def to_total(p):
    s = np.asarray(p["scores"])
    return {"scores": s, "probs": np.ones((len(s), 1)), "K": 1}
pan_preds = [to_total(p) for p in pan_src]
pan_gts   = [np.array([float(g.sum())]) for g in pan_gtc]

# NuInsSeg already total-count
nu_preds = dnu["preds"]
nu_gts   = dnu["gts"]

print(f"\nPanNuke (source): {len(pan_preds)} patches | GT total mean={np.mean([g[0] for g in pan_gts]):.1f}")
print(f"NuInsSeg (target): {len(nu_preds)} patches | GT total mean={np.mean([g[0] for g in nu_gts]):.1f}")

## 03 — Total-count nonconformity / interval (K=1)

In [ ]:
ALPHA = 0.1

def nonconf(p, gt):
    if len(p["scores"]) == 0: return float(abs(gt[0]))
    n  = pb_count(p["scores"], p["probs"])[0]
    sg = np.sqrt(pb_variance(p["scores"], p["probs"])[0] + 1e-6)
    return abs(gt[0] - n) / sg

def interval(p, q):
    if len(p["scores"]) == 0: return 0.0, 0.0
    n  = pb_count(p["scores"], p["probs"])[0]
    sg = np.sqrt(pb_variance(p["scores"], p["probs"])[0] + 1e-6)
    return max(0.0, n - q * sg), n + q * sg

def cov_width(preds, gts, q):
    los = np.array([interval(p, q)[0] for p in preds])
    his = np.array([interval(p, q)[1] for p in preds])
    g   = np.array([gg[0] for gg in gts])
    return float(np.mean((g >= los) & (g <= his))), float(np.mean(his - los))

## 04 — CROSS-DATASET: calibrate PanNuke -> test NuInsSeg

Split conformal (no adaptation) shows the honest coverage drop under shift.
Online methods warm-start on PanNuke, then stream NuInsSeg with feedback (5 stream seeds).

In [ ]:
# Calibrate quantile on ALL PanNuke
pan_scores = np.array([nonconf(pan_preds[i], pan_gts[i]) for i in range(len(pan_preds))])
q_cross = empirical_quantile(pan_scores, ALPHA)
print(f"q (calibrated on PanNuke) = {q_cross:.3f}")

# (A) Split conformal: fixed PanNuke quantile applied to NuInsSeg (no update)
split_cov, split_w = cov_width(nu_preds, nu_gts, q_cross)

# (B) Online methods: warm-start PanNuke, stream NuInsSeg with feedback
def stream(kind, nseeds=5):
    covs, ws = [], []
    for sd in range(nseeds):
        order = np.random.RandomState(sd).permutation(len(nu_preds))
        if kind == "aci":
            m = AdaptiveConformalInference(alpha_target=ALPHA, gamma=0.05)
            m.reset(); m.history_scores = list(pan_scores)
        else:
            m = PBAwareJointConformalOnline(alpha=ALPHA, window=300)
            m.warmstart(pan_scores)
        c, w = [], []
        for i in order:
            q = m.get_quantile(); lo, hi = interval(nu_preds[i], q)
            covered = lo <= nu_gts[i][0] <= hi
            c.append(covered); w.append(hi - lo)
            s = nonconf(nu_preds[i], nu_gts[i])
            m.update(s, covered) if kind == "aci" else m.update(s)
        covs.append(np.mean(c)); ws.append(np.mean(w))
    return float(np.mean(covs)), float(np.std(covs)), float(np.mean(ws)), float(np.std(ws))

aci_c, aci_cs, aci_w, aci_ws = stream("aci")
pbo_c, pbo_cs, pbo_w, pbo_ws = stream("pbo")

print("\nCROSS-DATASET (cal PanNuke -> test NuInsSeg):")
print(f"  Split (no adapt) : cov {split_cov*100:.1f}% | width {split_w:.2f}")
print(f"  ACI (stream)     : cov {aci_c*100:.1f}+/-{aci_cs*100:.1f}% | width {aci_w:.2f}+/-{aci_ws:.2f}")
print(f"  PB-JCI Online    : cov {pbo_c*100:.1f}+/-{pbo_cs*100:.1f}% | width {pbo_w:.2f}+/-{pbo_ws:.2f}")

## 05 — In-domain NuInsSeg reference (calibrate NuInsSeg, 5 seeds)

In [ ]:
def indomain_split(nseeds=5):
    covs, ws = [], []
    for sd in [42, 100, 200, 300, 400][:nseeds]:
        idx = np.random.RandomState(sd).permutation(len(nu_preds))
        ncal = len(idx) // 2
        cal, test = idx[:ncal], idx[ncal:]
        cs = np.array([nonconf(nu_preds[i], nu_gts[i]) for i in cal])
        q = empirical_quantile(cs, ALPHA)
        c, w = cov_width([nu_preds[i] for i in test], [nu_gts[i] for i in test], q)
        covs.append(c); ws.append(w)
    return float(np.mean(covs)), float(np.std(covs)), float(np.mean(ws)), float(np.std(ws))

id_c, id_cs, id_w, id_ws = indomain_split()
print(f"In-domain split (cal NuInsSeg): cov {id_c*100:.1f}+/-{id_cs*100:.1f}% | width {id_w:.2f}+/-{id_ws:.2f}")

## 06 — Summary table + save

In [ ]:
print("=" * 78)
print("CROSS-DATASET SHIFT: PanNuke (cal) -> NuInsSeg (test) | total count, alpha=0.1")
print("=" * 78)
print(f"{'Setting / Method':38s} | {'Coverage':>14s} | {'Width':>12s}")
print("-" * 78)
print(f"{'In-domain split (cal NuInsSeg)':38s} | {id_c*100:>6.1f}+/-{id_cs*100:<4.1f}% | {id_w:>7.2f}")
print(f"{'Cross split (cal PanNuke, no adapt)':38s} | {split_cov*100:>11.1f}% | {split_w:>7.2f}")
print(f"{'Cross ACI (stream feedback)':38s} | {aci_c*100:>6.1f}+/-{aci_cs*100:<4.1f}% | {aci_w:>7.2f}")
print(f"{'Cross PB-JCI Online (stream)':38s} | {pbo_c*100:>6.1f}+/-{pbo_cs*100:<4.1f}% | {pbo_w:>7.2f}")
print("-" * 78)
drop = (id_c - split_cov) * 100
print(f"\nCoverage DROP (in-domain -> cross split): {drop:+.1f} pp  "
      f"-> {'shift hurts split conformal' if drop > 1 else 'mild'}")

out = {
    "in_domain_split":  {"coverage": [id_c, id_cs], "width": [id_w, id_ws]},
    "cross_split":      {"coverage": split_cov, "width": split_w},
    "cross_aci":        {"coverage": [aci_c, aci_cs], "width": [aci_w, aci_ws]},
    "cross_pbjci_online": {"coverage": [pbo_c, pbo_cs], "width": [pbo_w, pbo_ws]},
    "q_cross": float(q_cross), "alpha": ALPHA,
}
with open("/kaggle/working/cross_dataset_results.json", "w") as f:
    json.dump(out, f, indent=2)
print("\nSaved: /kaggle/working/cross_dataset_results.json")

## Notes

- **Split (cal PanNuke, no adapt)** = honest coverage under real shift; expect < 90%.
- **ACI / PB-JCI Online** assume streaming GT feedback on NuInsSeg → recover coverage.
- Send `cross_dataset_results.json` to fill the cross-dataset row in PAPER_TABLES.